## 1. Text Model Training 1 (bert-base-multilingual-cased)

In [1]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "bert-base-multilingual-cased"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_bert_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_bert_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_bert_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_bert_results/confusion_matrix.png")

plt.show()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_730/3646957294.py:138: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting Text Model Training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 2. Text Model Training 2 (xlm-roberta-base) 

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "xlm-roberta-base"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_xlm_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_xlm_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_xlm_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_xlm_results/confusion_matrix.png")

plt.show()

## 3. Text Model Training 3 (distilbert-base-multilingual-cased)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "distilbert-base-multilingual-cased"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_distilbert_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_distilbert_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_distilbert_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_distilbert_results/confusion_matrix.png")

plt.show()

## 4. Text Model Training 4 (xlm-roberta-large)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "xlm-roberta-large"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_xlmr_large_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_xlmr_large_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_xlmr_large_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_xlmr_large_results/confusion_matrix.png")

plt.show()

## 5. Text Model Training 5 (mdeberta-v3-base)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "microsoft/mdeberta-v3-base"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_mdeberta_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_mdeberta_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_mdeberta_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_mdeberta_results/confusion_matrix.png")

plt.show()

## 6. Text Model Training 6 (xlm-mlm-100-1280)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "xlm-mlm-100-1280"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_xlm_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_xlm_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_xlm_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_xlm_results/confusion_matrix.png")

plt.show()

## 7. Text Model Training 7 (rembert)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "google/rembert"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_rembert_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_rembert_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_rembert_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_rembert_results/confusion_matrix.png")

plt.show()

## 8. Text Model Training 8 (IndicBERTv2-MLM-only)

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, confusion_matrix

# ----------------------------------------
# 1. Language Labels
# ----------------------------------------

languages = [
    "bengali","gujarati","hindi","kannada","malayalam",
    "marathi","odia","punjabi","sanskrit","tamil","telugu","urdu"
]

label2id = {l:i for i,l in enumerate(languages)}
id2label = {i:l for i,l in enumerate(languages)}

# ----------------------------------------
# 2. Model Name
# ----------------------------------------

MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

# ----------------------------------------
# 3. Tokenizer
# ----------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------
# 4. Dataset Class
# ----------------------------------------

class TextDataset(Dataset):

    def __init__(self, csv_file):

        self.data = pd.read_csv(csv_file)

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        text = str(row["transcription"])
        label = label2id[row["language"]]

        tokens = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": tokens["input_ids"].squeeze(),
            "attention_mask": tokens["attention_mask"].squeeze(),
            "labels": torch.tensor(label)
        }

# ----------------------------------------
# 5. Load Datasets
# ----------------------------------------

train_dataset = TextDataset("master_valid_augmented_cleaned.csv")

val_dataset = TextDataset(
    "master_test_known_dataset_standardized2.csv"
)

test_dataset = TextDataset(
    "master_test_unknown_dataset_standardized2.csv"
)

# ----------------------------------------
# 6. Load Model
# ----------------------------------------

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=12,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

# ----------------------------------------
# 7. Training Configuration
# ----------------------------------------

training_args = TrainingArguments(

    output_dir="./text_indicbertv2_results",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    num_train_epochs=50,

    learning_rate=3e-5,

    logging_steps=100,

    save_total_limit=2
)

# ----------------------------------------
# 8. Evaluation Metric
# ----------------------------------------

def compute_metrics(pred):

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(pred.label_ids, preds)
    }

# ----------------------------------------
# 9. Trainer
# ----------------------------------------

trainer = Trainer(

    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

# ----------------------------------------
# 10. Train Model
# ----------------------------------------

print("🚀 Starting Text Model Training...")

trainer.train()

# ----------------------------------------
# 11. Plot Loss Curve
# ----------------------------------------

logs = trainer.state.log_history

train_loss = []
epochs = []

for log in logs:

    if "loss" in log and "epoch" in log:
        train_loss.append(log["loss"])
        epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(train_loss, label="Training Loss")

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Text Model Training Loss Curve")

plt.legend()

plt.savefig("text_indicbertv2_results/loss_curve.png")

plt.show()

# ----------------------------------------
# 12. Accuracy Curve
# ----------------------------------------

eval_accuracy = []
eval_epochs = []

for log in logs:

    if "eval_accuracy" in log:
        eval_accuracy.append(log["eval_accuracy"])
        eval_epochs.append(log["epoch"])

plt.figure(figsize=(8,5))

plt.plot(eval_epochs, eval_accuracy, marker="o")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy Curve")

plt.savefig("text_indicbertv2_results/accuracy_curve.png")

plt.show()

# ----------------------------------------
# 13. Final Evaluation
# ----------------------------------------

print("\n🔥 Evaluating on Test Dataset...")

results = trainer.predict(test_dataset)

y_true = results.label_ids
y_pred = results.predictions.argmax(-1)

print("Final Text Model Accuracy:", accuracy_score(y_true, y_pred))

# ----------------------------------------
# 14. Confusion Matrix
# ----------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=languages,
    yticklabels=languages
)

plt.xlabel("Predicted Language")
plt.ylabel("True Language")
plt.title("Text Model Confusion Matrix")

plt.savefig("text_indicbertv2_results/confusion_matrix.png")

plt.show()